# Temel Transformer Mimarisi (PyTorch)

Bu notebook, "Attention Is All You Need" (Vaswani ve ark., 2017) makalesinde tanıtılan
Transformer mimarisinin sıfırdan, sade bir PyTorch implementasyonunu içerir.

İçerik:
1. Pozisyonel Kodlama (Positional Encoding)
2. Çok Başlı Dikkat Mekanizması (Multi-Head Attention)
3. İleri Beslemeli Ağ (Position-wise Feed Forward)
4. Encoder Katmanı
5. Decoder Katmanı
6. Tam Transformer Modeli (Encoder + Decoder)
7. Modeli test etme (ileri besleme / forward pass)
8. Basit bir "kopyalama" görevinde eğitim örneği


In [1]:
import math                          # Matematiksel işlemler (sqrt, log, vb.) için
import torch                         # PyTorch ana kütüphanesi
import torch.nn as nn                # Sinir ağı katmanları (Linear, Dropout, vb.)
import torch.nn.functional as F      # Aktivasyon fonksiyonları (softmax, relu) için

torch.manual_seed(42)                # Tekrarlanabilirlik için rastgelelik tohumu


## 1. Pozisyonel Kodlama (Positional Encoding)

Transformer, RNN'lerin aksine sıralamayı doğal olarak modellemez; tüm token'lara aynı
anda (paralel) bakar. Bu nedenle, kelimelerin dizideki **konumu** hakkında bilgi
modele ayrıca eklenmelidir. Bunun için sinüs ve kosinüs fonksiyonlarından oluşan
sabit bir pozisyonel kodlama matrisi kullanılır ve token embedding'lerine eklenir.

$$PE_{(pos, 2i)} = \\sin\\left(\\frac{pos}{10000^{2i/d_{model}}}\\right), \\quad
PE_{(pos, 2i+1)} = \\cos\\left(\\frac{pos}{10000^{2i/d_{model}}}\\right)$$


In [2]:
class PositionalEncoding(nn.Module):          # Pozisyonel kodlama modülü
    def __init__(self, d_model, max_len=5000): # d_model: embedding boyutu, max_len: maks. dizi uzunluğu
        super().__init__()
        pe = torch.zeros(max_len, d_model)     # (max_len, d_model) boyutunda sıfır matrisi
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1) pozisyon indeksleri
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))  # Frekans terimleri
        pe[:, 0::2] = torch.sin(position * div_term)   # Çift indekslere sinüs
        pe[:, 1::2] = torch.cos(position * div_term)   # Tek indekslere kosinüs
        self.register_buffer("pe", pe.unsqueeze(0))    # (1, max_len, d_model) - sabit olarak kaydet

    def forward(self, x):                      # x: (batch, seq_len, d_model)
        return x + self.pe[:, : x.size(1), :]  # Token embedding'ine pozisyon kodlamasını ekle


## 2. Çok Başlı Dikkat Mekanizması (Multi-Head Attention)

Dikkat (attention) mekanizmasının temel formülü:

$$\\text{Attention}(Q, K, V) = \\text{softmax}\\left(\\frac{QK^T}{\\sqrt{d_k}}\\right) V$$

Çok başlı (multi-head) versiyonda, $d_{model}$ boyutu $h$ başa bölünür, her baş
kendi alt-uzayında dikkat hesaplar ve sonuçlar birleştirilir. Bu, modelin diziyi
farklı açılardan (farklı ilişki türlerinden) eş zamanlı incelemesini sağlar.


In [3]:
class MultiHeadAttention(nn.Module):              # Çok başlı dikkat mekanizması
    def __init__(self, d_model, num_heads):       # d_model: giriş boyutu, num_heads: baş sayısı
        super().__init__()
        assert d_model % num_heads == 0            # d_model baş sayısına tam bölünmeli
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads            # Her başın alt-uzay boyutu

        self.W_q = nn.Linear(d_model, d_model)     # Query dönüşüm matrisi
        self.W_k = nn.Linear(d_model, d_model)     # Key dönüşüm matrisi
        self.W_v = nn.Linear(d_model, d_model)     # Value dönüşüm matrisi
        self.W_o = nn.Linear(d_model, d_model)     # Çıkış dönüşüm matrisi

    def split_heads(self, x):                      # Başlara ayır: (batch, seq, d_model) -> (batch, n_heads, seq, d_k)
        batch_size, seq_len, _ = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):                    # Başları birleştir: (batch, n_heads, seq, d_k) -> (batch, seq, d_model)
        batch_size, _, seq_len, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

    def forward(self, query, key, value, mask=None):  # Q, K, V ve isteğe bağlı maske
        Q = self.split_heads(self.W_q(query))      # Query'yi başlara ayır
        K = self.split_heads(self.W_k(key))        # Key'i başlara ayır
        V = self.split_heads(self.W_v(value))      # Value'yu başlara ayır

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # Ölçeklendirilmiş dikkat skorları
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))  # Maskelenmiş yerlere -inf

        attn = F.softmax(scores, dim=-1)           # Softmax ile ağırlıklar
        context = torch.matmul(attn, V)            # Ağırlıklı toplam (bağlam vektörü)
        output = self.W_o(self.combine_heads(context))  # Başları birleştir ve çıkışa yansıt
        return output, attn                       # Çıktı ve dikkat ağırlıkları


## 3. İleri Beslemeli Ağ (Position-wise Feed Forward)

Her dikkat bloğundan sonra, her pozisyona bağımsız olarak uygulanan basit bir
iki katmanlı MLP bulunur: genişlet (ReLU ile) → daralt.


In [4]:
class PositionwiseFeedForward(nn.Module):           # Pozisyon bazlı ileri beslemeli ağ
    def __init__(self, d_model, d_ff, dropout=0.1): # d_model: giriş/çıkış, d_ff: gizli katman boyutu
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)     # Genişletme katmanı (d_model -> d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)     # Daraltma katmanı (d_ff -> d_model)
        self.dropout = nn.Dropout(dropout)           # Düzenlileştirme (overfitting önleme)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))  # Linear -> ReLU -> Dropout -> Linear


## 4. Encoder Katmanı

Bir encoder katmanı şu sırayla çalışır:
self-attention → residual + layer norm → feed forward → residual + layer norm.


In [5]:
class EncoderLayer(nn.Module):                       # Bir encoder katmanı
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)    # Self-attention alt katmanı
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)  # FFN alt katmanı
        self.norm1 = nn.LayerNorm(d_model)           # Layer normalization 1
        self.norm2 = nn.LayerNorm(d_model)           # Layer normalization 2
        self.dropout = nn.Dropout(dropout)            # Dropout

    def forward(self, x, mask=None):                 # x: (batch, seq_len, d_model)
        attn_out, _ = self.self_attn(x, x, x, mask)  # Self-attention (Q=K=V=x)
        x = self.norm1(x + self.dropout(attn_out))   # Residual bağlantı + Dropout + LayerNorm
        ff_out = self.feed_forward(x)                # FFN'den geçir
        x = self.norm2(x + self.dropout(ff_out))     # Residual bağlantı + Dropout + LayerNorm
        return x


## 5. Decoder Katmanı

Decoder katmanı üç alt bloktan oluşur:
1. **Maskeli self-attention**: decoder'ın gelecekteki token'lara bakmasını önler.
2. **Cross-attention**: decoder, encoder çıkışına (kaynak diziye) bakar.
3. **Feed forward**: encoder ile aynı yapı.

Her alt blok residual bağlantı ve layer normalization ile sarılır.


In [6]:
class DecoderLayer(nn.Module):                       # Bir decoder katmanı
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)    # Maskeli self-attention
        self.cross_attn = MultiHeadAttention(d_model, num_heads)   # Cross-attention (encoder'a bakar)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)  # FFN
        self.norm1 = nn.LayerNorm(d_model)           # LayerNorm 1
        self.norm2 = nn.LayerNorm(d_model)           # LayerNorm 2
        self.norm3 = nn.LayerNorm(d_model)           # LayerNorm 3
        self.dropout = nn.Dropout(dropout)            # Dropout

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        attn_out, _ = self.self_attn(x, x, x, tgt_mask)  # Maskeli self-attention (geleceği görme)
        x = self.norm1(x + self.dropout(attn_out))   # Residual + Dropout + LayerNorm

        cross_out, _ = self.cross_attn(x, enc_output, enc_output, src_mask)  # Cross-attention (Q:decoder, K=V:encoder)
        x = self.norm2(x + self.dropout(cross_out))  # Residual + Dropout + LayerNorm

        ff_out = self.feed_forward(x)                # FFN'den geçir
        x = self.norm3(x + self.dropout(ff_out))     # Residual + Dropout + LayerNorm
        return x


## 6. Tam Transformer Modeli

Şimdi tüm parçaları birleştiriyoruz: kaynak/hedef embedding'leri, pozisyonel
kodlama, $N$ adet encoder katmanı, $N$ adet decoder katmanı ve son olarak
hedef kelime dağarcığı üzerinde tahmin yapan bir çıkış katmanı.

`generate_mask` fonksiyonu iki maske üretir:
- **src_mask**: padding (dolgu) token'larının dikkat hesabına katılmasını önler.
- **tgt_mask**: padding'i önler **ve** decoder'ın geleceği görmesini engelleyen
  (causal / look-ahead) üçgen maskeyi uygular.


In [7]:
class Transformer(nn.Module):                        # Tam Transformer modeli (Encoder-Decoder)
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8,
                 num_layers=6, d_ff=2048, max_len=100, dropout=0.1):
        super().__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)  # Kaynak token embedding
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)  # Hedef token embedding
        self.positional_encoding = PositionalEncoding(d_model, max_len) # Pozisyonel kodlama

        self.encoder_layers = nn.ModuleList(         # N adet encoder katmanı
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.decoder_layers = nn.ModuleList(         # N adet decoder katmanı
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)  # Çıkış projeksiyonu (vocab üzerinde logit)
        self.dropout = nn.Dropout(dropout)            # Dropout

    def generate_mask(self, src, tgt):               # Kaynak ve hedef maskelerini oluştur
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, src_len) padding maskesi
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)   # (batch, 1, tgt_len, 1) padding maskesi

        seq_len = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_len, seq_len), diagonal=1)).bool()  # Üçgen maske (geleceği gizle)
        tgt_mask = tgt_mask & nopeak_mask             # Padding + gelecek maskesini birleştir
        return src_mask, tgt_mask

    def encode(self, src, src_mask):                  # Encoder ileri beslemesi
        x = self.dropout(self.positional_encoding(self.encoder_embedding(src)))  # Embed + PosEncoding + Dropout
        for layer in self.encoder_layers:             # Her encoder katmanından geçir
            x = layer(x, src_mask)
        return x                                      # (batch, src_len, d_model)

    def decode(self, tgt, enc_output, src_mask, tgt_mask):  # Decoder ileri beslemesi
        x = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))  # Embed + PosEncoding + Dropout
        for layer in self.decoder_layers:             # Her decoder katmanından geçir
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x                                      # (batch, tgt_len, d_model)

    def forward(self, src, tgt):                      # İleri besleme (eğitim için)
        src_mask, tgt_mask = self.generate_mask(src, tgt)  # Maskeleri oluştur
        enc_output = self.encode(src, src_mask)       # Encoder çıktısı
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)  # Decoder çıktısı
        return self.fc_out(dec_output)                # Kelime dağarcığı üzerinde logit'ler


## 7. Modeli Test Etme

Modeli rastgele küçük bir girdiyle çalıştırıp çıkış boyutunu ve toplam parametre
sayısını kontrol edelim. Çıkış şekli `(batch, tgt_seq_len, tgt_vocab_size)`
olmalıdır; yani her hedef pozisyon için kelime dağarcığı üzerinde bir olasılık
dağılımı (logit) üretilir.


In [8]:
src_vocab_size = 1000                                # Kaynak kelime dağarcığı boyutu
tgt_vocab_size = 1000                                # Hedef kelime dağarcığı boyutu

model = Transformer(                                 # Model örneği oluştur
    src_vocab_size, tgt_vocab_size,
    d_model=128, num_heads=4, num_layers=2, d_ff=256, max_len=50
)

src = torch.randint(1, src_vocab_size, (2, 10))     # Rastgele kaynak dizi (batch=2, src_len=10)
tgt = torch.randint(1, tgt_vocab_size, (2, 12))     # Rastgele hedef dizi (batch=2, tgt_len=12)

out = model(src, tgt)                                # İleri besleme
print("Cikis sekli:", out.shape)                     # (batch, tgt_len, tgt_vocab_size)

total_params = sum(p.numel() for p in model.parameters())  # Toplam parametre sayısı
print(f"Toplam parametre sayisi: {total_params:,}")


Cikis sekli: torch.Size([2, 12, 1000])
Toplam parametre sayisi: 1,047,528


## 8. Basit Eğitim Örneği: "Kopyalama" Görevi

Modelin gerçekten öğrenip öğrenmediğini görmek için çok basit bir görev
kurguluyoruz: **girdi dizisini birebir kopyalamak**. Bu, eğitim döngüsünün
(kayıp fonksiyonu, optimizer, teacher forcing, maskeleme) nasıl bir araya
geldiğini göstermek için idealdir.

- `0` → padding (PAD)
- `1` → dizi başlangıcı (SOS)
- `2` → dizi sonu (EOS)
- `3..29` → gerçek "kelime" token'ları

Decoder girişi (`tgt_input`) hedefin bir adım kaydırılmış hali olup başına
SOS eklenmiştir (teacher forcing); kayıp ise gerçek hedef (`tgt_output`) ile
modelin tahminleri arasında hesaplanır.


In [9]:
VOCAB_SIZE = 30                                      # Küçük kelime dağarcığı (0-29)
SEQ_LEN = 8                                          # Dizi uzunluğu
SOS, EOS, PAD = 1, 2, 0                              # Özel token'lar: Başlangıç, Bitiş, Dolgu

def make_batch(batch_size=32, seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE):
    seq = torch.randint(3, vocab_size, (batch_size, seq_len))  # 3..29 arası rastgele token'lar
    src = seq                                                    # Kaynak = rastgele dizi
    tgt_full = torch.cat(                                        # Hedefe SOS başa, EOS sona ekle
        [torch.full((batch_size, 1), SOS), seq, torch.full((batch_size, 1), EOS)], dim=1
    )
    tgt_input = tgt_full[:, :-1]   # Decoder girişi: SOS ile başlar, son token hariç
    tgt_output = tgt_full[:, 1:]   # Gerçek hedef: EOS ile biter, ilk token (SOS) hariç
    return src, tgt_input, tgt_output


toy_model = Transformer(                             # Küçük bir Transformer modeli
    VOCAB_SIZE, VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2, d_ff=128, max_len=20
)
optimizer = torch.optim.Adam(toy_model.parameters(), lr=1e-3)  # Adam optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD)              # Kayıp: CrossEntropy (PAD'i yoksay)

losses = []                                                    # Kayıp değerlerini kaydetmek için
for epoch in range(300):                                       # 300 epoch eğitim
    src, tgt_input, tgt_output = make_batch()                   # Rastgele batch oluştur
    optimizer.zero_grad()                                       # Gradyanları sıfırla
    logits = toy_model(src, tgt_input)                          # Model tahmini
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_output.reshape(-1))  # Kayıp hesabı
    loss.backward()                                              # Geri yayılım
    optimizer.step()                                             # Ağırlıkları güncelle
    losses.append(loss.item())                                   # Kaybı kaydet
    if epoch % 50 == 0 or epoch == 299:                          # Her 50 epoch'ta yazdır
        print(f"Epoch {epoch:3d} | Kayip (loss): {loss.item():.4f}")


Epoch   0 | Kayip (loss): 3.5353


Epoch  50 | Kayip (loss): 2.6501


Epoch 100 | Kayip (loss): 1.6284


Epoch 150 | Kayip (loss): 0.7630


Epoch 200 | Kayip (loss): 0.4420


Epoch 250 | Kayip (loss): 0.3551


Epoch 299 | Kayip (loss): 0.2727


Kayıp (loss) değerinin epoch'lar boyunca düzenli şekilde düştüğünü
görmelisiniz — bu, modelin kopyalama görevini başarıyla öğrendiğinin bir
göstergesidir. İsterseniz aşağıdaki hücreyle eğitilmiş modeli tek bir örnek
üzerinde test edebilirsiniz (basit, greedy decoding).


In [10]:
@torch.no_grad()                                                     # Gradyan hesaplamasını devre dışı bırak (çıkarım)
def greedy_decode(model, src, max_len=SEQ_LEN + 2):                  # Açgözlü kod çözme (her adımda en olası token)
    model.eval()                                                      # Değerlendirme moduna geç
    src_mask, _ = model.generate_mask(src, src)                      # Kaynak maskesi
    enc_output = model.encode(src, src_mask)                         # Encoder çıktısını hesapla

    ys = torch.full((src.size(0), 1), SOS, dtype=torch.long)         # SOS token ile başlat
    for _ in range(max_len - 1):                                     # Adım adım token üret
        _, tgt_mask = model.generate_mask(src, ys)                    # Hedef maskesini oluştur
        out = model.decode(ys, enc_output, src_mask, tgt_mask)        # Decoder'dan geçir
        logits = model.fc_out(out[:, -1, :])                          # Son token'ın logit'leri
        next_token = logits.argmax(dim=-1, keepdim=True)              # En olası token'ı seç (argmax)
        ys = torch.cat([ys, next_token], dim=1)                       # Üretilen token'ı diziye ekle
    model.train()                                                     # Tekrar eğitim moduna dön
    return ys                                                         # (batch, üretilen_seq_len)


test_src, _, _ = make_batch(batch_size=1)                            # Tek örnekle test batch'i
prediction = greedy_decode(toy_model, test_src)                      # Model tahmini

print("Girdi dizisi   :", test_src.tolist()[0])                      # Kaynak dizi
print("Model ciktisi  :", prediction.tolist()[0])                    # Model çıktısı (SOS + kopya + EOS)


Girdi dizisi   : [19, 5, 26, 23, 29, 13, 6, 11]
Model ciktisi  : [1, 19, 5, 26, 23, 29, 13, 6, 11, 2]
